In [ ]:
import mujoco
import numpy as np
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import SubprocVecEnv
from stable_baselines3.common.callbacks import BaseCallback
from env import UnitreeA1Env
from pathlib import Path


In [ ]:
xmlPath = r"D:\Files\Scripts\py\Graduation Project\external\mujoco_menagerie\unitree_a1\scene.xml"
modelsPath = r"D:\Files\Scripts\py\Graduation Project\lab\Models"
if not Path(modelsPath).exists():
	Path(modelsPath).mkdir(parents=True, exist_ok=True)
videosPath = r"D:\Files\Scripts\py\Graduation Project\lab\Videos"
if not Path(videosPath).exists():
	Path(videosPath).mkdir(parents=True, exist_ok=True)

model = mujoco.MjModel.from_xml_path(xmlPath)
data = mujoco.MjData(model)

dt = float(model.opt.timestep)

In [ ]:
import multiprocessing

# See how many cores you have
n_cores = multiprocessing.cpu_count()
print(f"Available cores: {n_cores}")

# Rule of thumb: n_envs = number of physical cores
# Leave 1-2 cores free for the OS and main training thread
N_ENVS = n_cores - 2
print(f"Using {N_ENVS} envs")

Available cores: 16
Using 14 envs


In [ ]:
version = "1.1"

In [ ]:
# check if model version already exists
if Path(f"{modelsPath}/a1_walk_v{version}.zip").exists():
	response = input(f"Model [a1_walk_v{version}.zip] already exists. Overwrite? (y/n): ")
	if response.lower() != "y":
		print("Aborting training.")
		exit()


def make_env(xml):
    def _init():
        return UnitreeA1Env(xml)
    return _init

class LogCallback(BaseCallback):
    def _on_step(self):
        if len(self.model.ep_info_buffer) > 0 and self.n_calls % 10_000 == 0:
            mean_reward = np.mean([e["r"] for e in self.model.ep_info_buffer])
            mean_len    = np.mean([e["l"] for e in self.model.ep_info_buffer])
            print(f"steps={self.num_timesteps:>8} | mean_ep_reward={mean_reward:>8.2f} | mean_ep_len={mean_len:>6.0f}")
        return True

if __name__ == "__main__":
    n_envs = N_ENVS or multiprocessing.cpu_count() - 2
    print(f"Using {n_envs} envs for training")

    env = SubprocVecEnv([make_env(xmlPath) for _ in range(N_ENVS)])

    model = PPO(
        "MlpPolicy",
        env,
        n_steps=2048,
        batch_size=64 * n_envs,
        n_epochs=10,
        gamma=0.99,
        gae_lambda=0.95,
        clip_range=0.2,
        ent_coef=0.01,
        learning_rate=3e-4,
        verbose=0,
        tensorboard_log="./tb_logs/",
        policy_kwargs=dict(
            net_arch=[256, 256]  # bigger network than default [64, 64]
        )
    )

    model.learn(total_timesteps=10_000_000, callback=LogCallback())
    model.save(f"{modelsPath}/a1_walk_v{version}")
    env.close()